# Video → walkable 3D

Captures Studio flow, on **this** internship repo. One indoor walkthrough clip in. A Gaussian splat you can walk through out.

| Captures Studio | This notebook |
|---|---|
| 1. Capture a slow walkthrough | Phone video, overlapping motion, even light |
| 2. Drag and drop the clip | Cell **Upload** |
| 3. Cloud 3D engine | COLMAP + gsplat on this Colab GPU |
| 4–5. Walk and share | Download PLY → `python -m ui.splat_demo --job-id <job>` |

Works for **every** video you upload. The job name is the file name. Runtime → Change runtime type → **GPU** first.

Film with the Captures Studio routine before you upload: [smartphone video guide](https://www.captures.studio/blogs/how-to-create-3d-from-smartphone-video) and the [interactive tutorial](https://www.captures.studio/interactive-capture-tutorial). About five minutes, ultrawide, focus and exposure locked, three slow loops (chest, overhead, knee). The engine undistorts the phone lens, then trains 30,000 steps. Expect **about 2–3 hours** on a T4. A fast one-minute pan still comes out soft.

## 1. GPU and project code (run once per session)

In [ ]:
from google.colab import drive, files
import os, re, shutil, zipfile

drive.mount("/content/drive")
DRIVE_SPLAT = "/content/drive/MyDrive/indoor_splat"
os.makedirs(DRIVE_SPLAT, exist_ok=True)

print("Upload a zip of this project (exclude .venv). Needed so Colab uses the densify trainer.")
uploaded_repo = files.upload()
repo_zip = next(iter(uploaded_repo))
shutil.rmtree("/content/repo_unpack", ignore_errors=True)
zipfile.ZipFile(repo_zip).extractall("/content/repo_unpack")

def _find_pyproject(start):
    for dirpath, _, names in os.walk(start):
        if "pyproject.toml" in names and os.path.isdir(os.path.join(dirpath, "model")):
            return dirpath
    raise SystemExit("pyproject.toml not found in the zip")

src = _find_pyproject("/content/repo_unpack")
ROOT = "/content/ai-indoor-3d-reconstruction"
if os.path.abspath(src) != os.path.abspath(ROOT):
    shutil.rmtree(ROOT, ignore_errors=True)
    shutil.copytree(src, ROOT)
os.chdir(ROOT)
print("project", ROOT)

In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg
!pip -q install -e ".[splat]"
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU, then rerun"
print("engine ready on", torch.cuda.get_device_name(0))

## 2. Upload (every clip the user sends)

Select one video or many. Each file becomes its own scene. A zip of JPEGs is one scene.

In [ ]:
from google.colab import files
import os, re

print("Drop every indoor video to train. Each file is its own scene.")
uploaded = files.upload()
clips = []
for source in uploaded:
    stem = os.path.splitext(os.path.basename(source))[0]
    job = re.sub(r"[^a-zA-Z0-9_-]+", "_", stem).strip("_").lower() or "upload"
    clips.append({"source": source, "job": job})
    print("queued", job, os.path.getsize(source), "bytes")
print("scenes:", len(clips))

## 3. Process (3D engine)

Trains each uploaded video in order. One failure does not cancel the rest. Every 2000 steps that scene's point_cloud.ply is copied to its own Drive folder.

In [ ]:
import os, shutil, subprocess, sys, zipfile

os.environ["SPLAT_EXTRACT_FPS"] = "2"
os.environ["SPLAT_MAX_FRAMES"] = "180"
os.environ["SPLAT_IMAGE_MAX_SIZE"] = "1280"
os.environ["SPLAT_TRAIN_STEPS"] = "30000"
os.environ["SPLAT_MAX_GAUSSIANS"] = "800000"

for clip in clips:
    source = clip["source"]
    job = clip["job"]
    job_drive = os.path.join(DRIVE_SPLAT, job)
    os.makedirs(job_drive, exist_ok=True)
    os.environ["SPLAT_KEEP_DIR"] = job_drive
    print("training", job, "->", job_drive, flush=True)
    if source.lower().endswith(".zip"):
        dest = f"data/frames/{job}/splat"
        os.makedirs(dest, exist_ok=True)
        zipfile.ZipFile(source).extractall(dest)
        cmd = [sys.executable, "-m", "model.splat", f"data/frames/{job}", "--job-id", job]
    else:
        cmd = [sys.executable, "-m", "model.splat", source, "--job-id", job]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as exc:
        print("failed", job, exc, flush=True)
        continue
    ply = f"data/frames/{job}/splat/point_cloud.ply"
    if not os.path.isfile(ply):
        print("failed", job, "no ply", flush=True)
        continue
    size = os.path.getsize(ply)
    kept = os.path.join(job_drive, "point_cloud.ply")
    shutil.copy2(ply, kept)
    print("Drive", kept, size, "bytes", flush=True)
    if size < 5_000_000:
        print(job, "is still small. That clip did not have enough overlap. The other videos still train.", flush=True)

## 4. Experience (walk the scene)

Download the PLY, then on your Mac:

```bash
mkdir -p data/frames/<job>/splat
cp ~/Downloads/point_cloud.ply data/frames/<job>/splat/point_cloud.ply
python -m ui.splat_demo --job-id <job>
```

Click the view, then **WASD**. Each new upload is a new `<job>` folder, same as uploading another clip in Captures.

In [ ]:
from google.colab import files
for clip in clips:
    keep = os.path.join(DRIVE_SPLAT, clip["job"], "point_cloud.ply")
    if not os.path.isfile(keep):
        print("missing", clip["job"])
        continue
    print("download", clip["job"], os.path.getsize(keep), "bytes")
    files.download(keep)
print("scenes on Drive:", sorted(os.listdir(DRIVE_SPLAT)))